# Proyecto Integrador – Minería de Datos I

# Notebook 2
# Calidad y Limpieza de Datos

---

## Integrantes

- Marcia Lovaiza
- .....................................

## Materia

- Minería de Datos I

## Fecha

- 30-06-2026

##Sede

- Villa Ojo de Agua

---

# Introducción

Luego de la inspección inicial realizada en el Notebook 1, se procede a la etapa de preparación de datos.

En esta fase se corrigen inconsistencias, se normalizan categorías, se tratan valores faltantes y se documenta cada transformación realizada mediante un registro ETL, garantizando la trazabilidad y reproducibilidad del proceso.

# Objetivos

## Objetivo general

Mejorar la calidad del conjunto de datos para que pueda utilizarse en las etapas posteriores del proyecto.

## Objetivos específicos

- Detectar inconsistencias.
- Normalizar categorías.
- Convertir tipos de datos.
- Analizar valores faltantes.
- Revisar posibles valores atípicos.
- Documentar todas las transformaciones realizadas.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

# Carga del Dataset Original

In [ ]:
df = pd.read_json("../data/raw/streaming_users_dirty.json")

df.head()

# Registro ETL

Se crea una estructura que permitirá registrar todas las transformaciones realizadas sobre el conjunto de datos.

In [ ]:
def registrar(paso, descripcion, df):
    pipeline.append({
        "Paso": paso,
        "Descripción": descripcion,
        "Filas": len(df),
        "Nulos": int(df.isnull().sum().sum()),
        "Retención (%)": 100
    })

In [ ]:
registrar(
    "Inicio",
    "Carga del dataset original",
    df
)

##LIMPIEZA 1

# Normalización del Plan de Suscripción

Durante la inspección inicial se detectaron distintas formas de representar un mismo plan de suscripción.

Esta situación puede provocar categorías duplicadas durante el análisis.

In [ ]:
df["subscription_plan"].value_counts(dropna=False)

## Acción aplicada

Se unifican los nombres de los planes eliminando espacios, diferencias entre mayúsculas y minúsculas y distintas formas de escritura.

In [ ]:
df["subscription_plan"] = (
    df["subscription_plan"]
    .str.strip()
    .str.lower()
)

df["subscription_plan"] = (
    df["subscription_plan"]
    .replace({
        "std":"estándar",
        "estandar":"estándar",
        "basico":"básico",
        "básico":"básico",
        "premium":"premium"
    })
)

In [ ]:
df["subscription_plan"].value_counts(dropna=False)

## Interpretación

Luego de la normalización se observa una reducción de categorías duplicadas, mejorando la consistencia del conjunto de datos.

##Limpieza 2

# Normalización de Países

In [ ]:
df["country"].value_counts(dropna=False)

In [ ]:
df["country"] = (
    df["country"]
    .str.strip()
    .str.title()
)

df["country"] = (
    df["country"]
    .replace({
        "Brazil":"Brasil"
    })
)

In [ ]:
df["country"].value_counts(dropna=False)

Las categorías fueron unificadas para evitar diferencias ortográficas o de formato que pudieran afectar el análisis exploratorio.

##Limpieza 3

*** Genero***

In [ ]:
df["favorite_genre"].value_counts(dropna=False)

In [ ]:
df["favorite_genre"] = (
    df["favorite_genre"]
    .str.strip()
    .str.title()
)

df["favorite_genre"] = (
    df["favorite_genre"]
    .replace({
        "Comedy":"Comedia",
        "Documentary":"Documental",
        "Accion":"Acción"
    })
)

In [ ]:
df["favorite_genre"].value_counts(dropna=False)

## Limpieza 4

***Fechas***

In [ ]:
df["last_login_date"].head()

In [ ]:
df["last_login_date"] = pd.to_datetime(
    df["last_login_date"],
    errors="coerce"
)

In [ ]:
df["last_login_date"].head()

Las fechas fueron convertidas al formato datetime para facilitar su manipulación y análisis. Los valores inválidos se transformaron en NaT mediante el parámetro `errors="coerce"`.

## Limpieza 5

***Valores Faltantes***

In [ ]:
df.isnull().sum()

In [ ]:
df.fillna({
    "age":df["age"].median(),
    "subscription_plan":df["subscription_plan"].mode()[0],
    "country":df["country"].mode()[0],
    "favorite_genre":df["favorite_genre"].mode()[0],
    "customer_support_tickets":df["customer_support_tickets"].median()
}, inplace=True)

In [ ]:
df.isnull().sum()

Los valores faltantes fueron tratados utilizando la mediana para variables numéricas y la moda para variables categóricas, con el objetivo de preservar la distribución de los datos y minimizar el impacto sobre el análisis.

## Limpieza 6

***Valores Atípicos***

In [ ]:
df.describe()

Se revisaron las variables numéricas para identificar posibles valores extremos. En esta etapa no se eliminaron registros automáticamente, ya que primero se evaluó su posible impacto sobre el análisis.

In [ ]:
registrar(
    "Final",
    "Dataset limpio",
    df
)

In [ ]:
log = pd.DataFrame(pipeline)

log

In [ ]:
log.to_csv("../logs/pipeline_log.csv",index=False)

In [ ]:
df.to_json(
    "../data/processed/streaming_users_clean.json",
    orient="records",
    indent=2
)

# Conclusiones

Durante esta etapa se mejoró significativamente la calidad del conjunto de datos mediante la normalización de categorías, la conversión de tipos de datos y el tratamiento de valores faltantes.

Todas las transformaciones fueron documentadas mediante un registro ETL, garantizando la trazabilidad del proceso y generando un dataset limpio que será utilizado en el análisis exploratorio desarrollado en el siguiente notebook.